In [1]:
import cv2
import numpy as np


class OCRUtils:

    def __init__(self):
        pass

    # =====================================================
    # RECT CENTER
    # =====================================================

    def get_center_from_rect(self, rect):

        x, y, w, h = rect

        cx = x + w / 2
        cy = y + h / 2

        return cx, cy

    # =====================================================
    # GROUP CONTOURS HORIZONTAL
    # =====================================================

    def group_contours_into_lines_h(
        self,
        contours,
        x_threshold=20
    ):

        # bounding boxes
        rects = [(cv2.boundingRect(cnt), cnt)for cnt in contours]

        # centers
        rects_with_centers = [(rect,self.get_center_from_rect(rect),cnt)for rect, cnt in rects]

        # sort by x
        rects_with_centers.sort(key=lambda r: r[1][0])

        lines = []
        current_line = []

        for rect, (cx, cy), cnt in rects_with_centers:

            if not current_line:
                current_line.append(
                    (rect, cx, cy, cnt)
                )
                continue

            _, prev_cx, _, _ = current_line[-1]

            if abs(cx - prev_cx) < x_threshold:
                current_line.append(
                    (rect, cx, cy, cnt)
                )

            else:
                lines.append(current_line)

                current_line = [
                    (rect, cx, cy, cnt)
                ]

        if current_line:
            lines.append(current_line)

        result = []
        result_org = []

        for line in lines:

            line.sort(key=lambda r: r[2])

            result.append([
                rect for rect, _, _, _ in line
            ])

            result_org.append([
                cnt for _, _, _, cnt in line
            ])

        return result, result_org

    # =====================================================
    # GROUP CONTOURS VERTICAL
    # =====================================================

    def group_contours_into_lines_v(
        self,
        contours,
        y_threshold=20
    ):

        rects = [
            (cv2.boundingRect(cnt), cnt)
            for cnt in contours
        ]

        rects_with_centers = [
            (
                rect,
                self.get_center_from_rect(rect),
                cnt
            )
            for rect, cnt in rects
        ]

        rects_with_centers.sort(
            key=lambda r: r[1][1]
        )

        lines = []
        current_line = []

        for rect, (cx, cy), cnt in rects_with_centers:

            if not current_line:
                current_line.append(
                    (rect, cx, cy, cnt)
                )

                continue

            _, _, prev_cy, _ = current_line[-1]

            if abs(cy - prev_cy) < y_threshold:

                current_line.append(
                    (rect, cx, cy, cnt)
                )

            else:

                lines.append(current_line)

                current_line = [
                    (rect, cx, cy, cnt)
                ]

        if current_line:
            lines.append(current_line)

        result = []
        result_org = []

        for line in lines:

            line.sort(key=lambda r: r[1])

            result.append([
                rect for rect, _, _, _ in line
            ])

            result_org.append([
                cnt for _, _, _, cnt in line
            ])

        return result, result_org

    # =====================================================
    # GET HEADER COLUMN
    # =====================================================

    def get_heading_col(
        self,
        number_col,
        ans_org_v
    ):

        for i_cnt in range(len(ans_org_v)):

            if len(ans_org_v[i_cnt]) == number_col:

                return ans_org_v[i_cnt]

    # =====================================================
    # COUNTER MATCHING
    # =====================================================

    def counter_matching(
        self,
        matching_counters,
        counter_only,
        box_only,
        text_only
    ):

        box = []
        text = []

        for matching_counter in matching_counters:

            for index, counter in enumerate(counter_only):

                if (
                    matching_counter.shape == counter.shape
                    and np.all(matching_counter == counter)
                ):

                    box.append(box_only[index])

                    text.append(text_only[index])

        return box, text

    # =====================================================
    # COUNTER MATCHING HEADER
    # =====================================================

    def counter_matching_header(
        self,
        header_indexs,
        table_header,
        ans_org_h,
        counter_only,
        box_only,
        text_only
    ):

        matched_cnt = []

        for header_index in header_indexs:

            for index, counter_group in enumerate(ans_org_h):

                for counter in counter_group:

                    if (
                        table_header[header_index].shape
                        == counter.shape
                        and np.all(
                            table_header[header_index]
                            == counter
                        )
                    ):

                        matched_cnt.append(
                            ans_org_h[index]
                        )

        return matched_cnt

    # =====================================================
    # BOX CENTER
    # =====================================================

    def get_center(self, box):

        x_coords = [p[0] for p in box]
        y_coords = [p[1] for p in box]

        return (
            sum(x_coords) / 4,
            sum(y_coords) / 4
        )

    # =====================================================
    # GROUP OCR BOXES INTO LINES
    # =====================================================

    def group_boxes_into_lines(
        self,
        boxes,
        y_threshold=10
    ):

        box_centers = [
            (
                box,
                self.get_center(box),
                text,
                score
            )
            for box, text, score in boxes
        ]

        box_centers.sort(
            key=lambda b: b[1][1]
        )

        lines = []
        current_line = []

        for box, (cx, cy), text, score in box_centers:

            if score >= 0.9:

                if not current_line:

                    current_line.append(
                        (
                            box,
                            cx,
                            cy,
                            text,
                            score
                        )
                    )

                    continue

                _, _, prev_cy, _, _ = current_line[-1]

                if abs(cy - prev_cy) < y_threshold:

                    current_line.append(
                        (
                            box,
                            cx,
                            cy,
                            text,
                            score
                        )
                    )

                else:

                    lines.append(current_line)

                    current_line = [
                        (
                            box,
                            cx,
                            cy,
                            text,
                            score
                        )
                    ]

        if current_line:
            lines.append(current_line)

        sorted_lines = []
        sorted_lines_text = []

        for line in lines:

            line.sort(key=lambda b: b[2])

            sorted_lines.append([
                b[0] for b in line
            ])

            sorted_lines_text.append([
                b[3] for b in line
            ])

        return sorted_lines, sorted_lines_text

# import the libary

In [9]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os
from llama_cpp import Llama
import json

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# arranging the contours horizontally and vertically 

#### find the center of x and y

In [ ]:
def get_center_from_rect(rect):
    x, y, w, h = rect
    cx = x + w / 2
    cy = y + h / 2
    return cx, cy

#### grouping counter vertically 

In [ ]:
def group_contours_into_lines_h(contours, x_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][0])  # sort by cx

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cx
        _, prev_cx,_, _ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cx - prev_cx) < x_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[2])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

#### grouping counter horizontal

In [ ]:
def group_contours_into_lines_v(contours, y_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][1])  # sort by cy

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cy
        _, _, prev_cy,_ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cy - prev_cy) < y_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[1])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

In [ ]:
ans_h,ans_org_h = group_contours_into_lines_h(counter_only)
ans_v,ans_org_v = group_contours_into_lines_v(counter_only)

# getting the header of the table 

In [ ]:
def get_heading_col(number_col,ans_org_v):
    for i_cnt in range(len(ans_org_v)):
        if(len(ans_org_v[i_cnt]) == number_col):
            return ans_org_v[i_cnt]

In [ ]:
table_header=get_heading_col(6,ans_org_v)

In [ ]:
table_header

# match the table header with text and text box 

In [ ]:
def counter_matching (matching_counters,counter_only,box_only,text_only):
    box=[]
    text=[]
    for matching_counter in matching_counters:
        for index,counter in enumerate(counter_only):
            if (matching_counter.shape == counter.shape) and np.all(matching_counter == counter):
                box.append(box_only[index])
                text.append(text_only[index])
    return box,text
            
        

In [ ]:
matched_cnt=counter_matching(table_header,counter_only,box_only,text_only)

In [ ]:
matched_cnt[1]

In [ ]:
ans_org_v[0][1]

# header info extraction from vertical align 

In [ ]:
ans_org_v[0]

In [ ]:
def counter_matching_header(header_indexs,table_header,ans_org_h,counter_only,box_only,text_only):
    matched_cnt=[]
    for header_index in header_indexs:
        for index,counter_group in enumerate(ans_org_h):
            for counter in counter_group:
                if (table_header[header_index].shape == counter.shape) and np.all(table_header[header_index] == counter):
                    
                    matched_cnt.append(ans_org_h[index])

            #     box.append(box_only[index])
            #     text.append(text_only[index])
    return matched_cnt

In [ ]:
header_info=counter_matching_header([1,2,5],table_header,ans_org_h,counter_only,box_only,text_only)

In [ ]:
ref_info=counter_matching(header_info[0],counter_only,box_only,text_only)

In [ ]:
ref_info[1]

# arranging the text box of paddleocr horizontally

In [ ]:
def get_center(box):
    #[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    x_coords = [p[0] for p in box]
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4


def group_boxes_into_lines(boxes, y_threshold=10):
    # Step 1: compute centers
    box_centers = [(box, get_center(box),text,score) for box,text,score in boxes]

    # Step 2: sort by Y (top to bottom)
    box_centers.sort(key=lambda b: b[1][1])

    lines = []
    current_line = []

    for box, (cx, cy) ,text ,score in box_centers:
        if score >= 0.9:
            if not current_line:
                current_line.append((box, cx, cy, text, score))
                continue
    
            _, _, prev_cy, _, _ = current_line[-1]
    
            # Step 3: check if same line
            if abs(cy - prev_cy) < y_threshold:
                current_line.append((box, cx, cy, text, score))
            else:
                lines.append(current_line)
                current_line = [(box, cx, cy, text, score)]
        else:
            continue

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line by X (left to right)
    sorted_lines = []
    sorted_lines_text = []
    for line in lines:
        line.sort(key=lambda b: b[2])  # sort by center x
        sorted_lines.append([b[0] for b in line])
        sorted_lines_text.append([b[3] for b in line])

    return sorted_lines,sorted_lines_text

In [ ]:
poly,text = group_boxes_into_lines(ocr_result)